# Projet 8 - Construire et testez une infrastructure de données

## Mission

Vous êtes nouvellement embauché comme Data Engineer dans l’entreprise GreenAndCoop, un fournisseur coopératif français d'électricité d'origine renouvelable dans les Hauts-de-France. 

En tant que Data Engineer, votre travail est de fournir quotidiennement des données météorologiques de qualité aux Data Scientists pour leurs modèles de prévision. La prévision de demande est un enjeu stratégique  pour GreenCoop. Elle permet de  :

1. équilibrer le réseau. GreenCoop doit s’assurer que la production et la consommation d’électricité sont équilibrées en temps réel pour éviter des pénalités financières.

2. optimiser la production. Les sources d’énergie renouvelable, comme le solaire et l’éolien, sont intermittentes. En prévoyant la demande, GreenCoop peut planifier plus efficacement l’utilisation de ses ressources et maximiser l’efficacité de la production.

3. maîtriser ses coûts. Une prévision précise de la demande permet de réduire les coûts liés à l’achat d’électricité sur le marché de gros, en évitant les achats d’urgence à des prix élevés.

Pour gagner en précision dans leurs prévisions, les Data Scientists de GreenCoop ont lancé un nouveau projet qui s’appelle Forecast 2.0. L’objectif de celui-ci est d’intégrer de nouvelles sources de données dans leurs algorithmes comme celles issues de stations météorologiques semi-professionnelles. 

Vous êtes intégré à ce projet en tant que Data Engineer. Votre mission consiste à concevoir, implémenter et industrialiser un pipeline de données de type ELT permettant d’ingérer, transformer et fiabiliser ces nouvelles données météorologiques. Les données proviennent de différents réseaux (open data, stations InfoClimat, stations Weather Underground, etc.) et sont transmises à des fréquences variables (toutes les 10, 15 ou 30 minutes).

En consultant  vos mails ce matin, vous découvrez un message d’Ouly, la cheffe du projet Forecast 2.0 et membre de l’équipe Data Science :

De : Ouly

À : Moi

Objet : Forecast 2.0 - Intégration de nouvelles sources de données dans la BDD

Salut,

J'espère que tu vas bien !

Je suis ravie que tu nous rejoignes sur ce beau projet. Tu vas nous aider à améliorer nos algorithmes de prévision en les enrichissant de nouvelles sources de données. On avait besoin d’un bon Data Engineer pour mettre en place un pipeline fiable, automatisé et maintenable.

Suite à notre discussion, tu trouveras en pièces jointes les liens vers plusieurs sources de données que nous avons identifiées. Ces nouvelles sources vont nous aider. En effet, elles se trouvent dans des localisations où notre modèle de prévision de la demande d'électricité est moins performant faute de relevés précis par les stations météorologiques officielles. Ces sources sont le réseau InfoClimat et le réseau Weather Underground. 

Les formats des données sont hétérogènes. Nous souhaitons conserver toutes les informations des sources que je t’ai envoyées dans la base de données finale. Pour faciliter l’ingestion et la synchronisation de ces différentes sources, nous te recommandons de t’appuyer sur un outil d’intégration tel qu’Airbyte, afin de centraliser les flux vers PostgreSQL avant transformation.

Nous souhaitons conserver toutes les informations des sources que je t’ai envoyées dans la base de données finale. Il faut donc les structurer dans un schéma de données cohérent adapté à des usages analytiques et à nos besoins de modélisation.

Pour la transformation, la documentation et les contrôles de qualité des données, nous souhaitons nous appuyer sur DBT, afin de centraliser la logique de transformation directement dans la base de données, de versionner les modèles et de garantir la fiabilité des données mises à disposition.

Je ne sais pas si tu es au courant, mais notre DSI nous impose de travailler avec AWS. Nous souhaitons que ces données soient stockées dans une base de données PostgreSQL hébergée sur AWS afin qu’elles puissent être facilement exploitées par les équipes Data Science, notamment dans nos environnements de machine learning (pour info, on utilise SageMaker pour nos travaux de ML donc on pourra se connecter directement).

Nous avons une réunion de l’équipe projet dans 1 mois. Peux-tu préparer une présentation pour nous expliquer ta démarche ainsi qu’une démonstration de la solution mise en place ? 

Les éléments qui nous intéressent sont les suivants :

* Le schéma de la base de données, et la manière dont les données peuvent être interrogées,

* le processus de collecte, de transformation et de test des données, notamment via DBT (on utilise souvent des logigrammes pour présenter nos processus),

* L’architecture globale de la base de données,

* La stack technique utilisée (les outils/services web utilisés),

* Les mécanismes de contrôle de la qualité des données (taux d’erreurs),

* Le délai de mise à disposition des données.

Bon courage à toi et à très bientôt,

Ouly

P.J: Liens vers les sources de données identifiées

1. Stations météorologiques du réseau InfoClimat (Bergues, Hazebrouck, Armentières, Lille-Lesquin)

2. Station amateur Weather Underground à Ichtegem, Belgique

3. Station amateur Weather Underground à La Madeleine, France



## Préparez votre environnement de travail

Docker compose pour le déployement de PostgreSQL. Voici le contenu du compose.yaml:

```
services:

  postgres:
    image: postgres:18
    container_name: postgres
    restart: unless-stopped
    environment:
      POSTGRES_USER: postgres
      POSTGRES_PASSWORD: Seb+postgresql1

    ports:
      - "5433:5432" # en local, le port 5432 est déjà utilisé par PostgreSQL
    volumes:
      - "./postgres_data:/var/lib/postgresql:rw"

  pgadmin:
     image: dpage/pgadmin4
     environment:
       - PGADMIN_DEFAULT_EMAIL=admin@admin.com
       - PGADMIN_DEFAULT_PASSWORD=root
     ports:
       - "8080:80"

# Airbyte déployé depuis Docker Compose est déprécié depuis 2024 au profit d’abctl (basé sur Docker). abctl est installé en local.
  # airbyte:
    # image: airbyte/airbyte:latest
    # container_name: airbyte
    # restart: unless-stopped
    # ports: 
      # - "8000:8000"
    # depends_on:
       #- postgres
```





`docker compose up`

Vérification de la base PostgreSQL.

```
(base) C:\Users\sebas>docker exec -it postgres psql -U postgres
psql (18.4 (Debian 18.4-1.pgdg13+1))
Type "help" for help.

postgres=# SELECT current_database();
 current_database
------------------
 postgres
(1 row)

``` 
Lancement de pgAdmin via http://localhost:8080/ dans la barre de l'explorateur internet


Récupération de la version du abclt sur le site `https://docs.airbyte.com/platform/using-airbyte/getting-started/oss-quickstart`

```
(base) C:\Users\sebas>abctl version

(base) C:\Users\sebas>abctl local install
```
Récupérer le mot de passe pour accéder à Airbyte.
```
(base) C:\Users\sebas>abctl local credentials
  INFO    Using Kubernetes provider:
            Provider: kind
            Kubeconfig: C:\Users\sebas\.airbyte\abctl\abctl.kubeconfig
            Context: kind-airbyte-abctl
 SUCCESS  Retrieving your credentials from 'airbyte-auth-secrets'
  INFO    Credentials:
            Email: [not set]
            Password: PQ3Lc9U0E5g4gdNZuiyOwni437dV4oJv
            Client-Id: 01670c34-d77a-4b64-94c4-9004ff627116
            Client-Secret: AJOV3rBUkRby8fK9wj69LR4a8RrtL6If
```
Rajouter une adresse mail (il n'est pas nécessaire qu'elle soit réelle pour un exercice).
```
(base) C:\Users\sebas>abctl local credentials --email sebastien@exemple.com
  INFO    Using Kubernetes provider:
            Provider: kind
            Kubeconfig: C:\Users\sebas\.airbyte\abctl\abctl.kubeconfig
            Context: kind-airbyte-abctl
  INFO    Updating email for authentication
 SUCCESS  Email updated
 SUCCESS  Retrieving your credentials from 'airbyte-auth-secrets'
  INFO    Credentials:
            Email: sebastien@exemple.com
            Password: PQ3Lc9U0E5g4gdNZuiyOwni437dV4oJv
            Client-Id: 01670c34-d77a-4b64-94c4-9004ff627116
            Client-Secret: AJOV3rBUkRby8fK9wj69LR4a8RrtL6If
```

lancer airbyte en tapant dans la barre de l'explorateur: `http://localhost:8000/`

si cela ne fonctionne pas, faire :`abctl local install --insecure-cookies` et relancer.

Se connecter avec l'email



## Récupérer les données météorologiques avec Airbyte

### Créer la database dans PostgreSQL en passant par pgAdmin.

![creation_db](create_database_pgadmin.png)


### Configurer la destination PostgreSQL dans Airbyte

Dans Airbyte, cliquer sur `Destinations` puis `+ New destination` puis `PostgreSQL`

Compléter les champs suivants:

* Destination name: Postgres
* Host : host.docker.internal
* Port : 5433
* Database : weather_data
* Schema : raw
* Username/Password : les identifiants de connexion à PostgreSQL définit dans le docker compose
* SSL Mode : disable

![configuration_destination](config_destination_postgres.png)


La destination a été créée avec succès:


![destination_creee](destination_creee.png)

### Configurer la source dans Airbyte

Avec abctl on ne peut pas simplement pointer vers un dossier local Windows pour la source des fichiers. 

On peut, via GitHub, mettre à disposition les fichiers Excel et JSON et laisser Airbyte les récupérer avec une URL raw.

Exemple:

`https://github.com/Sebules/Project_8_OpenClassrooms_Data_Engineer/raw/refs/heads/main/donnees/Weather_Underground_Ichtegem_BE.xlsx`

Création d'une source:

![config_source_excel.png](config_source_excel.png)


La source a été créée:


![source_excel_creee.png](source_excel_creee.png)

### Créer la connexion

cliquer sur `Create a connection`

![click_add_connection.png](click_add_connection.png)

Ensuite, cliquer sur `Postgres`

![click_postgres.png](click_postgres.png)


Choisir le sync mode full refresh orverwrite pour garder les données raw

![resultat_chose_sync_mode_full_refresh_overwrite.png](resultat_chose_sync_mode_full_refresh_overwrite.png)


Choisir schedule type manual

![resultat_chose_schedule_manual.png](resultat_chose_schedule_manual.png)


Cliquer sur `set up connection`. Le résultat en image.

![resultat_set_up_connection.png](resultat_set_up_connection.png)


Il faut faire la synchronisation:

![click_sync_now.png](click_sync_now.png)

Résultat de synchro qui s'est bien passée:

![click_sync_now_resultat.png](click_sync_now_resultat.png)


### Vérification dans PostgreSQL via pgAdmin

```
SET search_path TO raw;

SELECT * FROM "Weather_Underground_Ichtegem_BE" LIMIT 10;
```

![verification_table_belgique.png](verification_table_belgique.png)


## Transformez les données météorologiques avec DBT


source: https://docs.getdbt.com/docs/local/install-dbt?version=2.0

Installation de DBT en local:

`pip install dbt-core`

Vérification de l'installation:

```
(base) C:\Users\sebas>dbt --version
Core:
  - installed: 1.11.12
  - latest:    1.11.12 - Up to date!

Plugins:

``` 

source: https://docs.getdbt.com/docs/local/connect-data-platform/postgres-setup?version=2.0&name=Fusion

Il est nécessaire d'installer le package `dbt-postgre` pour pouvoir connecter PostgreSQL à DBT. La commande à taper est:

`pip install dbt-postgre`

Vérification de l'installation:

```
(base) C:\Users\sebas>dbt --version
Core:
  - installed: 1.11.12
  - latest:    1.11.12 - Up to date!

Plugins:
  - postgres: 1.10.2 - Up to date!
```





#### Initialisation du projet DBT

Lors que l'on utilise dbt Core, la création du fichier `profiles.yml` est nécessaire. 

source: 
* https://docs.getdbt.com/docs/local/profiles.yml?version=2.0
* https://docs.getdbt.com/docs/local/connect-data-platform/postgres-setup?version=2.0&name=Fusion

Création du projet "greencoop_projet" en tapant les commandes suivantes:

```
(base) C:\Users\sebas\Documents\cours Openclassrooms\Data Engineer\projet 8>dbt init greencoop_projet
20:06:17  Running with dbt=1.11.12
20:06:17  Creating dbt configuration folder at C:\Users\sebas\.dbt
20:06:17
Your new dbt project "greencoop_projet" was created!

For more information on how to configure the profiles.yml file,
please consult the dbt documentation here:

  https://docs.getdbt.com/docs/configure-your-profile

One more thing:

Need help? Don't hesitate to reach out to us via GitHub issues or on Slack:

  https://community.getdbt.com/

Happy modeling!

20:06:17  Setting up your profile.
Which database would you like to use?
[1] postgres

(Don't see the one you want? https://docs.getdbt.com/docs/available-adapters)

Enter a number:

```
Compléter le questionnaire ainsi:
```
(Don't see the one you want? https://docs.getdbt.com/docs/available-adapters)

Enter a number: 1
host (hostname for the instance): 127.0.0.1
port [5432]: 5433
user (dev username): postgres
pass (dev password):
dbname (default database that dbt will build objects in): weather_data
schema (default schema that dbt will build objects in): analytics
threads (1 or more) [1]: 4
21:17:18  Profile greencoop_projet written to C:\Users\sebas\.dbt\profiles.yml using target's profile_template.yml and your supplied values. Run 'dbt debug' to validate the connection.
```


Voici le contenu du profiles.yml

```
greencoop_projet:
  outputs:
    dev:
      dbname: weather_data
      host: 127.0.0.1
      pass: Seb+postgresql1
      port: 5433
      schema: analytics
      threads: 4
      type: postgres
      user: postgres
  target: dev
```

Dans le répertoire où l'initialisation a été lancée, DBT va créer un dossier avec le nom du projet, ici "greencoop_projet".

![dossier_projet_dbt](dossier_projet_dbt.png)



En se placant dans le répertoire du projet DBT, vérifier la connexion avec la base de données en utilisant la commande `dbt debug`:

```
(base) C:\Users\sebas\Documents\cours Openclassrooms\Data Engineer\projet 8>cd greencoop_projet

(base) C:\Users\sebas\Documents\cours Openclassrooms\Data Engineer\projet 8\greencoop_projet>dbt debug
07:10:56  Running with dbt=1.11.12
07:10:56  dbt version: 1.11.12
07:10:56  python version: 3.12.13
07:10:56  python path: C:\Users\sebas\anaconda3\python.exe
07:10:56  os info: Windows-11-10.0.26200-SP0
07:10:56  Using profiles dir at C:\Users\sebas\.dbt
07:10:56  Using profiles.yml file at C:\Users\sebas\.dbt\profiles.yml
07:10:56  Using dbt_project.yml file at C:\Users\sebas\Documents\cours Openclassrooms\Data Engineer\projet 8\greencoop_projet\dbt_project.yml
07:10:56  adapter type: postgres
07:10:56  adapter version: 1.10.2
07:10:56  Configuration:
07:10:56    profiles.yml file [OK found and valid]
07:10:56    dbt_project.yml file [OK found and valid]
07:10:56  Required dependencies:
07:10:56   - git [OK found]

07:10:56  Connection:
07:10:56    host: 127.0.0.1
07:10:56    port: 5433
07:10:56    user: postgres
07:10:56    database: weather_data
07:10:56    schema: analytics
07:10:56    connect_timeout: 10
07:10:56    role: None
07:10:56    search_path: None
07:10:56    keepalives_idle: 0
07:10:56    sslmode: None
07:10:56    sslcert: None
07:10:56    sslkey: None
07:10:56    sslrootcert: None
07:10:56    application_name: dbt
07:10:56    retries: 1
07:10:56  Registered adapter: postgres=1.10.2
07:10:57    Connection test: [OK connection ok]

07:10:57  All checks passed!

(base) C:\Users\sebas\Documents\cours Openclassrooms\Data Engineer\projet 8\greencoop_projet>
```

### Déclaration des sources DBT

https://docs.getdbt.com/best-practices?version=2.0

Afin de suivre les bonnes pratiques DBT, dans le dossier models du projet DBT, on structure le dossier en créant les sous-dossiers suivants:

* Staging
* Intermediate
* Marts

![](17682190270109_P1C3-b.png)

voir la source https://docs.getdbt.com/best-practices/how-we-structure/1-guide-overview?version=2.0 pour plus de détails.

On crée donc le fichier __sources.yml avec les informations suivantes:

```
version: 2

sources:
  - name: weather_data
    schema: raw
    tables:
      - name: Weather_Underground_Ichtegem_BE
      - name: Weather_Underground_La_Madeleine_FR
      - name: station_meteo_infoclimat

```


### Couche staging

Pour chaque table, le fichier stg_xxx.sql est créé. Exemple pour la table Weather_Underground_Ichtegem_BE:

```
{{config(materialized='view')}}
SELECT
    "UV" AS uv_index,
    REGEXP_REPLACE("Gust", '[^0-9\.\-]', '', 'g')::NUMERIC AS gust_mph,
    "Time"::TIME AS observation_time,
    LOWER("Wind") AS wind_direction,
    REGEXP_REPLACE("Solar", '[^0-9\.\-]', '', 'g')::NUMERIC AS solar_wm2,
    REGEXP_REPLACE("Speed", '[^0-9\.\-]', '', 'g')::NUMERIC AS speed_mph,
    REGEXP_REPLACE("Humidity", '[^0-9\.\-]', '', 'g')::NUMERIC AS humidity_pct,
    REGEXP_REPLACE("Pressure", '[^0-9\.\-]', '', 'g')::NUMERIC AS pressure_in,
    REGEXP_REPLACE("Dew_Point",'[^0-9\.\-]', '', 'g')::NUMERIC AS dew_point_f,
    REGEXP_REPLACE("Temperature", '[^0-9\.\-]', '', 'g')::NUMERIC AS temperature_f,
    REGEXP_REPLACE("Precip__Rate_", '[^0-9\.\-]', '', 'g')::NUMERIC AS precip_rate_in,
    REGEXP_REPLACE("Precip__Accum_", '[^0-9\.\-]', '', 'g')::NUMERIC AS precip_accum_in
FROM {{source('weather_data','Weather_Underground_Ichtegem_BE')}}

```


```
(base) C:\Users\sebas\Documents\cours Openclassrooms\Data Engineer\projet 8\greencoop_projet>dbt run --select stg_Weather_Underground_Ichtegem_BE
13:19:32  Running with dbt=1.11.12
13:19:32  Registered adapter: postgres=1.10.2
13:19:33  Found 3 models, 4 data tests, 3 sources, 476 macros
13:19:33
13:19:33  Concurrency: 4 threads (target='dev')
13:19:33
13:19:34  1 of 1 START sql view model analytics.stg_Weather_Underground_Ichtegem_BE ...... [RUN]
13:19:34  1 of 1 OK created sql view model analytics.stg_Weather_Underground_Ichtegem_BE . [CREATE VIEW in 0.23s]
13:19:34
13:19:34  Finished running 1 view model in 0 hours 0 minutes and 1.05 seconds (1.05s).
13:19:34
13:19:34  Completed successfully
13:19:34
13:19:34  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 TOTAL=1

```

Vérification dans pgAdmin que la view a bien été créée:

![verif_pgadmin](view_stg_weather_underground_be.png)
